# Insurance Policy PDF: Retrieval + Structured Extraction

**Goal:** upload an insurance PDF and get back a clean, structured summary (policy name, coverage benefits, exclusions, waiting periods, claim requirements) as **JSON** and **CSV**.

**How it works, in two parts**

| Part | Steps | What happens |
|---|---|---|
| A. Ingestion and retrieval | 1 to 8 | PDF is read, cut into chunks, converted to vectors, stored in a vector database and searched by meaning. |
| B. Structured extraction | 9 to 16 | For each field, the best-matching chunks are fetched and an LLM turns them into structured JSON. |

Run the cells from top to bottom. Use a GPU runtime (T4) for Part B.

## Part A: Ingestion and retrieval
### Step 1: Install libraries
Installs everything the notebook needs in one go:
- `pypdf` reads text out of the PDF.
- `langchain-*` provides loaders, the chunk splitter and the vector store wrapper.
- `sentence-transformers` turns text into embedding vectors.
- `chromadb` is the vector database.
- `transformers` and `accelerate` run the LLM used in Part B.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters \
    chromadb sentence-transformers pypdf transformers accelerate pandas

### Step 2: Imports
Loads the libraries used in Part A. Keeping imports together makes it easy to see what the pipeline depends on.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader        # PDF -> text, one document per page
from langchain_text_splitters import RecursiveCharacterTextSplitter  # text -> chunks
from langchain_huggingface import HuggingFaceEmbeddings              # text -> vectors
from langchain_community.vectorstores import Chroma                  # vector database

### Step 3: Upload the PDF
Opens a file picker in Colab. The uploaded file is saved in the Colab session and `pdf_path` stores its name for the later steps.

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Using:", pdf_path)

### Step 4: Load the PDF
`PyPDFLoader` reads the PDF and returns **one document per page**. Each document holds the page text and metadata (file name and page number). The page number is what lets us cite where an extracted fact came from.

If the page count is right but the text is empty, the PDF is probably a scan and needs OCR first.

In [ ]:
documents = PyPDFLoader(pdf_path).load()

print(f"Pages loaded: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:500])

### Step 5: Split the text into chunks
A whole page is too large and too mixed to search precisely, so each page is cut into smaller **chunks**.

- `chunk_size=1000` is the maximum characters per chunk. This is large enough to keep a full clause or list together. Smaller sizes (200 to 500) find more precise passages but can break lists apart.
- `chunk_overlap=50` repeats a little text between neighbouring chunks, so a sentence cut at the boundary is not lost.
- `separators` makes the splitter prefer natural breaks (paragraph, then line, then sentence, then word).

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "],
)

chunks = text_splitter.split_documents(documents)
print(f"Chunks created: {len(chunks)}")
print(chunks[0].page_content[:300])

### Step 6: Create the embedding model
An **embedding** converts text into a list of numbers that captures its *meaning*. Texts with similar meaning end up close together, so a search for "waiting period" can find "period before cover starts" even though the words differ.

`BAAI/bge-small-en-v1.5` is small, free and strong at retrieval. `normalize_embeddings=True` is recommended for BGE models so similarity scores are consistent.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model loaded")

### Step 7: Store the chunks in a vector database
Every chunk is embedded and saved in **Chroma**. Searching later means embedding the question and returning the chunks whose vectors are closest to it.

`persist_directory` saves the database to disk (`chroma_db`), so it can be reloaded without re-embedding while the Colab session is alive.

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db",
)
print("Vectors stored:", vectorstore._collection.count())

### Step 8: Test retrieval
Before building on top of retrieval, check that it returns sensible passages. The helper below takes a question, fetches the `k` most similar chunks and prints each with its page number.

`k` controls how many chunks come back: a small `k` is precise but may miss information; a large `k` covers more but adds noise.

In [ ]:
def retrieve_context(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    for i, d in enumerate(docs, 1):
        print(f"--- Chunk {i} (page {d.metadata.get('page', 0) + 1}) ---")
        print(d.page_content[:400].strip(), "\n")
    return docs

_ = retrieve_context("What is the waiting period for insurance claims?")

## Part B: Structured extraction
Retrieval finds the right passages, but they are still free text. Part B turns them into structured fields.

```
Field queries -> vector search -> relevant chunks -> LLM (JSON-only prompt) -> parse + validate -> JSON / CSV
```

### Step 9: Load the LLM
Loads `Qwen2.5-3B-Instruct`, an instruction-following model that fits the free T4 GPU in fp16. The `generate` function sends a system + user prompt and returns the reply. It uses **greedy decoding** (`do_sample=False`), so the same input gives the same output, which matters for extraction.

To use a different model, change `LLM_NAME` (for example `Qwen2.5-1.5B-Instruct` if memory is tight).

In [ ]:
import json, re, csv, time, hashlib
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU available:", torch.cuda.is_available())   # Runtime > Change runtime type > T4 GPU

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
llm.eval()


def generate(system_prompt, user_prompt, max_new_tokens=700):
    """Run one chat turn with greedy decoding and return only the new text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic output
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("LLM loaded:", LLM_NAME)

### Step 10: Define what to extract (the schema)
This dictionary is the **only place you edit** to extract different information. Each field has:
- `kind`: the output shape (`string`, `list_of_strings` or `list_of_objects`).
- `queries`: several ways the document might phrase this topic. They are all searched, because insurers word things differently.
- `instruction`: what the LLM should pull out of the retrieved text.

In [ ]:
FIELD_SPECS = {
    "policy_name": {
        "kind": "string",
        "queries": [
            "name of the insurance policy or product",
            "policy wordings title UIN plan name",
            "this policy is issued by the insurer",
        ],
        "instruction": (
            "Extract the official name of the insurance policy / product exactly as written "
            "(for example the title on the first page). Return a single string."
        ),
    },
    "coverage_benefits": {
        "kind": "list_of_objects",
        "keys": ["benefit", "details"],
        "queries": [
            "what benefits are covered under the policy",
            "coverage hospitalization expenses room rent ICU",
            "pre-hospitalization post-hospitalization day care procedures",
            "sum insured cumulative bonus ambulance AYUSH maternity",
        ],
        "instruction": (
            "List every benefit or coverage that the policy states it WILL pay for. "
            "For each one give a short 'benefit' name and a 'details' string with the limits, "
            "amounts, percentages or conditions mentioned (empty string if none)."
        ),
    },
    "exclusions": {
        "kind": "list_of_strings",
        "queries": [
            "what is not covered exclusions",
            "the company shall not be liable to make any payment",
            "permanent exclusions list",
            "excluded expenses non-payable items",
        ],
        "instruction": (
            "List every exclusion: situations, conditions, treatments or expenses the policy "
            "states it will NOT cover. One short string per exclusion."
        ),
    },
    "waiting_periods": {
        "kind": "list_of_objects",
        "keys": ["type", "duration", "applies_to"],
        "queries": [
            "waiting period",
            "initial waiting period 30 days",
            "pre-existing diseases waiting period months years",
            "specific disease waiting period after policy inception",
        ],
        "instruction": (
            "List every waiting period. For each give 'type' (e.g. initial, pre-existing disease, "
            "specific illness, maternity), 'duration' copied exactly as written (e.g. '30 days', "
            "'36 months'), and 'applies_to' (what it applies to; empty string if unclear)."
        ),
    },
    "claim_settlement_requirements": {
        "kind": "list_of_strings",
        "queries": [
            "claim settlement process procedure",
            "documents required for claim submission",
            "notice of claim intimation time limit hours days",
            "cashless claim reimbursement claim steps",
        ],
        "instruction": (
            "List the requirements a policyholder must meet to make or settle a claim: "
            "intimation/notice deadlines, documents to submit, cashless vs reimbursement steps, "
            "and the insurer's settlement timelines. One short string per requirement."
        ),
    },
}

print("Fields to extract:", list(FIELD_SPECS))

### Step 11: Retrieval and cleanup helpers
Four small helpers support the extraction:
- `retrieve_for_field` runs all of a field's queries, removes duplicate chunks and sorts them by page.
- `build_context` joins the chunks into one text block labelled `[Page N]`, capped at 7000 characters so the prompt fits the model.
- `parse_json_loose` safely reads the LLM reply even when it has code fences, extra words, trailing commas or was cut off mid-answer.
- `normalise` forces the parsed result into the exact shape the schema expects and removes empty or repeated items.

LLMs rarely return perfectly clean JSON, so this step is what makes the pipeline reliable.

In [ ]:
MAX_CONTEXT_CHARS = 7000   # keeps the prompt well inside the model's context window


def retrieve_for_field(field, k_per_query=4):
    """Retrieve, de-duplicate and page-sort the chunks relevant to one field."""
    spec = FIELD_SPECS[field]
    seen, collected = set(), []

    for q in spec["queries"]:
        for doc in vectorstore.similarity_search(q, k=k_per_query):
            key = hashlib.md5(doc.page_content.encode()).hexdigest()
            if key not in seen:
                seen.add(key)
                collected.append(doc)

    collected.sort(key=lambda d: d.metadata.get("page", 0))
    return collected


def build_context(docs, extra_first_page=False):
    """Format chunks as '[Page N] text' blocks, capped at MAX_CONTEXT_CHARS."""
    parts = []
    if extra_first_page:
        # The policy name nearly always sits on page 1
        parts.append("[Page 1]\n" + documents[0].page_content[:1500])

    for d in docs:
        parts.append(f"[Page {d.metadata.get('page', 0) + 1}]\n{d.page_content}")

    context = "\n\n---\n\n".join(parts)
    return context[:MAX_CONTEXT_CHARS]


def pages_of(docs):
    return sorted({d.metadata.get("page", 0) + 1 for d in docs})


def parse_json_loose(text):
    """Best-effort JSON parsing for LLM output. Returns a dict or None."""
    if not text:
        return None

    # Remove markdown code fences
    text = re.sub(r"```(?:json)?", "", text).strip()
    decoder = json.JSONDecoder()

    # 1) Decode the outer JSON object (first '{'); later '{' only count if they hold "result"
    starts = [m.start() for m in re.finditer(r"\{", text)]
    for n, st in enumerate(starts):
        try:
            obj, _ = decoder.raw_decode(text[st:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and (n == 0 or "result" in obj):
            return obj

    # 2) Repair truncated output: cut back to the last COMPLETE item, then close brackets
    start = text.find("{")
    if start != -1:
        body = text[start:].rstrip()
        cut_points = [i for i, ch in enumerate(body) if ch in ",}"]
        for pos in reversed(cut_points[-60:]):
            head = body[:pos + 1] if body[pos] == "}" else body[:pos]
            for closer in ("]}", "}"):
                try:
                    obj = json.loads(head + closer)
                    if isinstance(obj, dict):
                        return obj
                except json.JSONDecodeError:
                    continue
    return None


def _clean(s):
    return re.sub(r"\s+", " ", str(s)).strip() if s is not None else ""


def normalise(spec, parsed):
    """Coerce parsed LLM output into the exact shape declared in the spec."""
    if parsed is None:
        return None
    if "result" in parsed:
        raw = parsed["result"]
    elif len(parsed) == 1:                  # model used another wrapper key, e.g. {"items": [...]}
        raw = next(iter(parsed.values()))
    else:
        raw = parsed
    kind = spec["kind"]

    if kind == "string":
        if isinstance(raw, list):
            raw = raw[0] if raw else ""
        if isinstance(raw, dict):
            raw = next(iter(raw.values()), "")
        value = _clean(raw)
        return value if value.lower() not in {"", "null", "none", "n/a", "not found"} else None

    if not isinstance(raw, list):
        raw = [raw] if raw else []

    out, seen = [], set()
    if kind == "list_of_strings":
        for item in raw:
            if isinstance(item, dict):
                item = " - ".join(_clean(v) for v in item.values() if _clean(v))
            item = _clean(item)
            if item and item.lower() not in seen:
                seen.add(item.lower())
                out.append(item)
        return out

    if kind == "list_of_objects":
        keys = spec["keys"]
        for item in raw:
            if isinstance(item, str):
                item = {keys[0]: item}
            if not isinstance(item, dict):
                continue
            row = {k: _clean(item.get(k, "")) for k in keys}
            ident = " | ".join(row.values()).lower()
            if row[keys[0]] and ident not in seen:
                seen.add(ident)
                out.append(row)
        return out

    return None


print("Helpers defined")

### Step 12: Prompt and per-field extraction
`extract_field` handles one field from start to finish: retrieve chunks, build the prompt, call the LLM, parse the answer.

The system prompt tells the model to use **only the supplied context**, never invent details, and answer with JSON only. If the first reply cannot be parsed, it retries once with a stricter instruction. Each field is extracted in a separate call because small models do better on one task at a time.

Each result records its `status` (`ok`, `empty` or `parse_failed`) and the source pages.

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise information-extraction engine for insurance policy documents. "
    "Use ONLY the context provided. Never use outside knowledge and never invent details. "
    "If the information is not in the context, return an empty result. "
    "Reply with a single valid JSON object and nothing else - no explanations, no markdown."
)


def schema_example(spec):
    kind = spec["kind"]
    if kind == "string":
        return '{"result": "<string or null>"}'
    if kind == "list_of_strings":
        return '{"result": ["<string>", "<string>"]}'
    keys = spec["keys"]
    obj = "{" + ", ".join(f'"{k}": "<string>"' for k in keys) + "}"
    return '{"result": [' + obj + ", " + obj + "]}"


def build_user_prompt(field, context, strict=False):
    spec = FIELD_SPECS[field]
    extra = (
        "\nIMPORTANT: your previous reply was not valid JSON. "
        "Output ONLY the JSON object, starting with { and ending with }."
        if strict else ""
    )
    return (
        f"TASK: {spec['instruction']}\n\n"
        f"OUTPUT FORMAT (JSON only):\n{schema_example(spec)}\n\n"
        f"RULES:\n"
        f"- Keep each item short and factual; copy numbers and durations exactly as written.\n"
        f"- Do not repeat items.\n"
        f"- If nothing relevant is in the context, return an empty list (or null for a string).\n"
        f"{extra}\n"
        f"CONTEXT:\n{context}\n\n"
        f"JSON:"
    )


def extract_field(field, verbose=True):
    """Retrieve context for one field, call the LLM, parse and validate the result."""
    spec = FIELD_SPECS[field]
    t0 = time.time()

    docs = retrieve_for_field(field)
    context = build_context(docs, extra_first_page=(field == "policy_name"))

    raw = generate(SYSTEM_PROMPT, build_user_prompt(field, context))
    value = normalise(spec, parse_json_loose(raw))

    if value is None:   # one retry with a stricter prompt
        raw = generate(SYSTEM_PROMPT, build_user_prompt(field, context, strict=True))
        value = normalise(spec, parse_json_loose(raw))

    status = "ok" if value not in (None, "", []) else "empty"
    if value is None:
        value = "" if spec["kind"] == "string" else []
        status = "parse_failed"

    if verbose:
        n = value if isinstance(value, str) else f"{len(value)} items"
        print(f"  {field:<32} {status:<12} {n if isinstance(n, str) and len(n) < 60 else n}  ({time.time()-t0:.1f}s)")

    return {
        "value": value,
        "source_pages": pages_of(docs) if field != "policy_name" else sorted(set(pages_of(docs)) | {1}),
        "status": status,
        "method": "llm",
    }


print("Extractor ready")

### Step 13: Rule-based cross-check for waiting periods
Waiting periods follow simple patterns ("30 days", "36 months"), so a regular expression can find them directly in the page text without any LLM. It acts as a **safety net**: if the LLM returns no waiting periods, these regex matches are used instead and marked `regex_fallback` in the output.

In [ ]:
DURATION = r"(?:\d+|one|two|three|four|five|thirty|sixty|ninety)\s*(?:\(\d+\)\s*)?(?:day|month|year)s?"
WAITING_RE = re.compile(rf"[^.\n]*waiting\s+period[^.\n]*?(?P<dur>{DURATION})[^.\n]*", re.IGNORECASE)


def regex_waiting_periods(docs):
    """Scan every page for sentences that mention a waiting period with a duration."""
    hits, seen = [], set()
    for d in docs:
        page = d.metadata.get("page", 0) + 1
        text = re.sub(r"[ \t]+", " ", d.page_content)
        for m in WAITING_RE.finditer(text):
            sentence = _clean(m.group(0))
            key = sentence.lower()
            if key in seen or len(sentence) < 15:
                continue
            seen.add(key)
            hits.append({
                "type": "waiting period",
                "duration": _clean(m.group("dur")),
                "applies_to": sentence[:200],
                "page": page,
            })
    return hits


print("Regex cross-check ready")

### Step 14: Run the extraction
Runs Steps 11 to 13 for every field (about five LLM calls) and prints the result as JSON. The output has one entry per field with its value, source pages, status and the method used.

In [ ]:
def extract_policy_info(verbose=True):
    if verbose:
        print("Extracting fields...\n")

    result = {"source_file": os.path.basename(pdf_path), "pages_in_document": len(documents)}
    fields = {name: extract_field(name, verbose=verbose) for name in FIELD_SPECS}

    # Rule-based cross-check / fallback for waiting periods
    regex_hits = regex_waiting_periods(documents)
    wp = fields["waiting_periods"]
    wp["regex_matches"] = regex_hits
    if not wp["value"] and regex_hits:
        wp["value"] = [{k: h[k] for k in ("type", "duration", "applies_to")} for h in regex_hits]
        wp["source_pages"] = sorted({h["page"] for h in regex_hits})
        wp["method"] = "regex_fallback"
        wp["status"] = "ok"

    result["extracted"] = fields
    return result


extraction = extract_policy_info()

# Compact view: values only
print(json.dumps({k: v['value'] for k, v in extraction['extracted'].items()}, indent=2, ensure_ascii=False))

### Step 15: Export to JSON and CSV
- **JSON** keeps the full structure, including page references, status and method per field. Use it for programs and APIs.
- **CSV** is a flat table with one row per extracted item (category, item, details, source pages). Use it in Excel or Google Sheets.

The second cell downloads both files to your computer.

In [ ]:
def to_rows(result):
    """Flatten the nested result into one row per extracted item."""
    ex = result["extracted"]
    policy = ex["policy_name"]["value"]
    rows = []

    def add(category, item, details, field):
        rows.append({
            "policy_name": policy,
            "category": category,
            "item": item,
            "details": details,
            "source_pages": ", ".join(map(str, ex[field]["source_pages"])),
            "method": ex[field]["method"],
        })

    for b in ex["coverage_benefits"]["value"]:
        add("Coverage benefit", b["benefit"], b["details"], "coverage_benefits")
    for e in ex["exclusions"]["value"]:
        add("Exclusion", e, "", "exclusions")
    for w in ex["waiting_periods"]["value"]:
        add("Waiting period", w["type"], f"{w['duration']} - {w['applies_to']}".strip(" -"), "waiting_periods")
    for c in ex["claim_settlement_requirements"]["value"]:
        add("Claim requirement", c, "", "claim_settlement_requirements")

    return rows


base = os.path.splitext(os.path.basename(pdf_path))[0]
json_path = f"{base}_extracted.json"
csv_path = f"{base}_extracted.csv"

# JSON
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(extraction, f, indent=2, ensure_ascii=False)

# CSV (utf-8-sig so Excel shows special characters correctly)
rows = to_rows(extraction)
df = pd.DataFrame(rows, columns=["policy_name", "category", "item", "details", "source_pages", "method"])
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"Saved: {json_path}")
print(f"Saved: {csv_path}  ({len(df)} rows)")
df.head(20)

In [ ]:
# Download both files to your computer
from google.colab import files

files.download(json_path)
files.download(csv_path)

### Step 16: Quality check
Extraction should never be trusted blindly. This table shows, for each field, its status, how many items were found, which method produced them and which pages they came from. Any field marked `empty` or `parse_failed` is listed under "Fields needing attention".

**If a field is empty or incomplete:** raise `k_per_query` in `retrieve_for_field` (try 6 to 8), add more phrasings to that field's `queries`, or switch to a larger model.

In [ ]:
print(f"{'Field':<34}{'Status':<14}{'Items':<8}{'Method':<16}Pages")
print("-" * 90)
for name, data in extraction["extracted"].items():
    v = data["value"]
    n = 1 if (isinstance(v, str) and v) else (0 if isinstance(v, str) else len(v))
    pages = ", ".join(map(str, data["source_pages"][:8])) + (" ..." if len(data["source_pages"]) > 8 else "")
    print(f"{name:<34}{data['status']:<14}{n:<8}{data['method']:<16}{pages}")

problems = [k for k, v in extraction["extracted"].items() if v["status"] != "ok"]
print("\nFields needing attention:", problems if problems else "none")

### Limitations
- Scanned PDFs need OCR before Step 4.
- Very long lists (for example 40+ exclusions) may exceed the 7000-character context cap. Raise `MAX_CONTEXT_CHARS` or split the field into sub-queries.
- The checks verify the output's *shape*, not its truth. Use the page numbers to spot-check important values against the PDF.